# ♻️ Recycling Warehouse: Tile Surface Area Analysis & Batch Approval

Welcome to this project notebook! You will build an end-to-end vision pipeline that automatically **segments intact floor tiles and calculates usable surface area** to determine batch approval — a core task in recycling warehouse operations.

**The Problem**

A batch of reclaimed floor tiles arrives at the warehouse. Photos are taken. Your system must answer two questions:
1. *Where* are the intact (usable) regions? → **Segmentation** (U-Net)
2. *Is there enough usable surface area?* → **Binary Classification** (approve/reject based on 40% threshold)

**Pipeline Overview**

1. 🗂️ **Setup** — Clone repo, configure paths for Colab
2. 🔍 **Dataset** — Load 5000 image–mask pairs, visualize them
3. 🧱 **U-Net** — Train the segmentation backbone
4. 🔢 **FNN Classifier** — Classify from a pooled latent vector
5. 🧠 **CNN Classifier** — Classify from the full spatial latent map
6. ⚖️ **Comparison** — FNN vs CNN: table and metrics
7. 🚀 **Inference** — Walk through the full pipeline on one image

**Business Logic:**
- Each batch contains multiple reclaimed floor tiles photographed together
- U-Net segments intact (usable) regions from damaged/unusable areas
- Batches with **>40% intact surface area** → **APPROVE** (sell to customers)
- Batches with **<40% intact surface area** → **REJECT** (send for manual inspection)

By the end you will understand why spatial information in the bottleneck matters for surface area classification, and how two architecturally different heads can exploit the same U-Net backbone in different ways.

---
## 1 🗂️ Setup

Clone the project repository and add it to the Python path so the dataset can be accessed. All model code is defined directly in this notebook — no external module imports needed.

In [ ]:
# Run this cell once on Colab to clone the repository
!git clone -b week2/warehouse-manager/demo https://github.com/eth-bmai-fs26/project.git
%cd project

In [ ]:
import os
import sys

# Make sure the repo root is on the Python path so local imports work
PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.ndimage import label as scipy_label
import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

### 1.1 Utility Functions

Before defining our models, we need a helper function:

1. **Dice Loss** — A segmentation-specific loss that directly measures overlap between the predicted mask and the ground truth. Unlike Binary Cross-Entropy (which treats every pixel independently), Dice Loss considers the entire region as a whole. This makes it especially useful when the foreground/background class balance is uneven.

   $$\text{Dice} = \frac{2 \cdot |P \cap T|}{|P| + |T|}$$

   We return $1 - \text{Dice}$ so that **lower is better** (standard loss convention).

In [ ]:
def dice_loss(pred, target, eps=1e-5):
    """
    Compute the Dice Loss between predicted and target masks.
    
    The Dice coefficient measures the overlap between two binary masks:
        Dice = 2 * |intersection| / (|pred| + |target|)
    
    We apply sigmoid to the raw predictions first (logits → probabilities),
    then compute Dice and return 1 - Dice so it acts as a loss.
    
    Args:
        pred:   Raw model output (logits), shape (B, 1, H, W)
        target: Ground-truth binary mask, shape (B, 1, H, W)
        eps:    Small constant to avoid division by zero
    
    Returns:
        Scalar tensor: 1 - Dice coefficient (lower is better)
    """
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return 1 - (2 * intersection + eps) / (union + eps)

---
## 2 🔍 Dataset

We have **5000 floor tile batch images**, each paired with a binary segmentation mask that labels the intact (usable) regions.

The files follow this naming convention:

| Original image | Segmentation mask |
|---|---|
| `dataset_new/original/mosaic_0001.jpg` | `dataset_new/segmented/mosaic_0001-segmented.jpg` |
| `dataset_new/original/mosaic_0002.jpg` | `dataset_new/segmented/mosaic_0002-segmented.jpg` |
| … | … |
| `dataset_new/original/mosaic_5000.jpg` | `dataset_new/segmented/mosaic_5000-segmented.jpg` |

`SegmentationDataset` handles this via a `_mask_name()` helper that appends `-segmented` to the original stem. Both images are converted to **grayscale** and resized to **256×256**. Mask pixels above 0.5 are set to 1 (intact region), the rest to 0.

**Dataset composition:**
- **15% pristine batches** — nearly perfect tiles (high approval rate)
- **15% disaster batches** — heavily damaged (high rejection rate)  
- **70% standard batches** — typical mixed quality (where the 40% threshold matters most)

#### 2.1 Dataset Class

`SegmentationDataset` is a custom PyTorch `Dataset` that:
1. Scans the `original/` folder for all `.jpg` images
2. For each image, derives the mask filename by appending `-segmented` (e.g., `mosaic_0001.jpg` → `mosaic_0001-segmented.jpg`)
3. Loads both as **grayscale**, resizes to the target size (256×256)
4. **Binarises** the mask: any pixel above 0.5 becomes 1 (intact), the rest become 0 (damaged)

`LabeledSegmentationDataset` is a thin wrapper that also returns the approval label alongside each `(image, mask)` pair. This is essential after applying `random_split`, because subset indices no longer correspond to the original dataset order.

In [ ]:
class SegmentationDataset(Dataset):
    """
    PyTorch Dataset for paired (image, mask) segmentation data.
    
    Expects the following directory layout:
        root/
            original/       ← raw grayscale images
            segmented/      ← binary segmentation masks (suffix: -segmented)
    
    Each image is converted to grayscale, resized, and normalised to [0, 1].
    Masks are binarised at 0.5: pixels above → 1 (intact), below → 0 (damaged).
    """
    
    def __init__(self, root, image_size=(256, 256)):
        self.root = root
        self.image_size = image_size
        self.images = sorted(os.listdir(os.path.join(root, "original")))
        self.transform = T.Compose([
            T.Resize(image_size),
            T.ToTensor(),
        ])
    
    @staticmethod
    def _mask_name(img_name):
        """Derive the mask filename from the image filename.
        
        Example: 'mosaic_0001.jpg' → 'mosaic_0001-segmented.jpg'
        """
        stem, ext = os.path.splitext(img_name)
        return f"{stem}-segmented{ext}"
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = self._mask_name(img_name)
        
        # Load as grayscale (L mode) and apply transforms
        img = Image.open(os.path.join(self.root, "original", img_name)).convert("L")
        mask = Image.open(os.path.join(self.root, "segmented", mask_name)).convert("L")
        
        img = self.transform(img)
        mask = self.transform(mask)
        
        # Binarise the mask: intact regions → 1, damaged → 0
        mask = (mask > 0.5).float()
        
        return img, mask


class LabeledSegmentationDataset(Dataset):
    """
    Wraps a SegmentationDataset to also return the approval label.
    
    This is essential when using random_split: the Subset indices
    no longer match the original dataset order, so we bundle the
    label directly with each sample to avoid index mismatches.
    
    Returns: (image, mask, label) tuples
        - label = 1 → APPROVE (coverage ≥ 40%)
        - label = 0 → REJECT  (coverage < 40%)
    """
    
    def __init__(self, base_dataset, labels):
        self.base_dataset = base_dataset
        self.labels = labels
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        img, mask = self.base_dataset[idx]
        label = self.labels[idx]
        return img, mask, label

In [ ]:
DATA_ROOT = "week2/project1/dataset_new"

# ── 1. Load ground-truth labels from CSV ──────────────────────────────
ground_truth = pd.read_csv(f"{DATA_ROOT}/ground_truth.csv")
approval_labels = (ground_truth["status"] == "APPROVE").astype(int).tolist()

# ── 2. Build the labelled dataset ─────────────────────────────────────
ds_base = SegmentationDataset(DATA_ROOT, image_size=(256, 256))
ds_full = LabeledSegmentationDataset(ds_base, approval_labels)

# ── 3. Train / Validation / Test split (70 / 15 / 15) ────────────────
train_size = int(0.70 * len(ds_full))
val_size   = int(0.15 * len(ds_full))
test_size  = len(ds_full) - train_size - val_size

torch.manual_seed(42)                      # reproducibility
ds_train, ds_val, ds_test = random_split(ds_full, [train_size, val_size, test_size])

# ── 4. DataLoaders (each batch → (images, masks, labels)) ────────────
dl_train = DataLoader(ds_train, batch_size=32, shuffle=True)
dl_val   = DataLoader(ds_val,   batch_size=32, shuffle=False)
dl_test  = DataLoader(ds_test,  batch_size=32, shuffle=False)

print(f"Total dataset size: {len(ds_full):,} samples")
print(f"  Training set:     {len(ds_train):,} samples (70%)")
print(f"  Validation set:   {len(ds_val):,} samples (15%)")
print(f"  Test set:         {len(ds_test):,} samples (15%)")
print(f"\nEach batch yields: (images, masks, labels)")
print(f"  Image shape: {ds_base[0][0].shape}   (C, H, W)")
print(f"  Mask  shape: {ds_base[0][1].shape}   (C, H, W)")

In [ ]:
# Show label distribution
print(f"Total samples: {len(approval_labels)}")
print(f"Approved batches: {sum(approval_labels)} ({100*sum(approval_labels)/len(approval_labels):.1f}%)")
print(f"Rejected batches: {len(approval_labels)-sum(approval_labels)} ({100*(1-sum(approval_labels)/len(approval_labels)):.1f}%)")

# Visualize a sample of 8 batches from the test set
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
test_indices = list(ds_test.indices[:8])  # First 8 from test set

for col, idx in enumerate(test_indices):
    img, mask, label = ds_full[idx]
    status = "APPROVE" if label == 1 else "REJECT"
    coverage = ground_truth.iloc[idx]["coverage_pct"] * 100
    
    axes[0, col].imshow(img.squeeze(), cmap="gray")
    axes[0, col].set_title(f"mosaic_{idx+1:04d}\n{status} ({coverage:.1f}%)", fontsize=8)
    axes[0, col].axis("off")
    axes[1, col].imshow(mask.squeeze(), cmap="gray")
    axes[1, col].set_title("GT mask", fontsize=8)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("Mask", fontsize=11)
plt.suptitle("Sample from test set (held-out data)", fontsize=13)
plt.tight_layout()
plt.show()

#### What to look for

- Each mask should show **white regions** (value 1) over the intact/usable tiles and **black** (value 0) over damaged/unusable areas.
- **APPROVE** batches should have >40% white pixels (intact coverage)
- **REJECT** batches should have <40% white pixels
- These 8 samples are from the **test set** (held-out data not used for training)

---
## 3 🧱 U-Net Segmentation Model

The segmentation backbone is a **U-Net** — an encoder-decoder architecture with skip connections. Skip connections route fine-grained spatial detail from the encoder directly to the decoder, allowing the model to produce sharp, pixel-accurate masks even after aggressive downsampling.

**Architecture (`base_ch = 32`, input `256×256`)**

| Stage | Module | Output shape |
|---|---|---|
| Input | — | (B, 1, 256, 256) |
| Encoder block 1 | `ConvBlock(1 → 32)` | (B, 32, 256, 256) |
| Downsample 1 | `Conv2d stride 2` | (B, 32, 128, 128) |
| Encoder block 2 | `ConvBlock(32 → 64)` | (B, 64, 128, 128) |
| Downsample 2 | `Conv2d stride 2` | (B, 64, 64, 64) |
| **Bottleneck** | `ConvBlock(64 → 64)` | **(B, 64, 64, 64)** |
| Upsample 1 | `ConvTranspose2d` + skip from enc2 | (B, 32, 128, 128) |
| Upsample 2 | `ConvTranspose2d` + skip from enc1 | (B, 32, 256, 256) |
| Output | `Conv2d(32 → 1)` | (B, 1, 256, 256) |

The **bottleneck** is the key representation layer. Two methods expose it:
- `get_latent(x)` → applies Global Average Pooling → vector **(B, 64)**
- `get_latent_map(x)` → returns the raw spatial map **(B, 64, 64, 64)**

These two views of the bottleneck feed the FNN and CNN classifiers respectively.

**For our tile scenario:**  
The U-Net learns to segment intact (usable) tile regions. The white pixels in the output mask represent surface area that can be sold to customers.

#### 3.1 Architecture Definition

**`ConvBlock`** — the building block used in every encoder/decoder stage: two rounds of `Conv2d → BatchNorm → ReLU`. Batch normalisation stabilises training by normalising activations within each mini-batch.

**`UNet`** — the full encoder-decoder with:
- **Encoder**: Two ConvBlocks with stride-2 convolutions in between (halving spatial dimensions)
- **Bottleneck**: A ConvBlock that produces the latent representation
- **Decoder**: Transposed convolutions (upsampling) with skip connections from the encoder
- Two accessor methods: `get_latent()` (pooled vector) and `get_latent_map()` (spatial feature map)

In [ ]:
class ConvBlock(nn.Module):
    """
    Double convolution block: Conv → BN → ReLU → Conv → BN → ReLU
    
    This is the fundamental building block of the U-Net. Each stage in both
    the encoder and decoder uses one ConvBlock. The two convolutions allow
    the network to learn increasingly complex features at each spatial scale.
    
    Batch Normalisation after each conv stabilises training by re-centering
    and re-scaling activations, enabling higher learning rates.
    """
    
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    U-Net: Encoder-Decoder with skip connections for semantic segmentation.
    
    Architecture (base_ch=32, input 256×256):
        Encoder:
            enc1: ConvBlock(1 → 32)      → (B, 32, 256, 256)
            down1: Conv2d stride 2        → (B, 32, 128, 128)
            enc2: ConvBlock(32 → 64)      → (B, 64, 128, 128)
            down2: Conv2d stride 2        → (B, 64,  64,  64)
        
        Bottleneck:
            ConvBlock(64 → 64)            → (B, 64,  64,  64)  ← latent map
            Global Average Pooling        → (B, 64)             ← latent vector
        
        Decoder:
            up1: ConvTranspose2d + skip   → (B, 32, 128, 128)
            up2: ConvTranspose2d + skip   → (B, 32, 256, 256)
            out: Conv2d(32 → 1)           → (B,  1, 256, 256)
    
    The bottleneck representation is exposed via two methods:
        - get_latent(x)     → pooled vector  (B, latent_dim)     → for FNN classifier
        - get_latent_map(x) → spatial map    (B, latent_dim, H', W') → for CNN classifier
    """
    
    def __init__(self, in_ch=1, base_ch=32):
        super().__init__()
        self.latent_dim = base_ch * 2  # 64 channels at the bottleneck
        
        # ── Encoder ───────────────────────────────────────────────────
        self.enc1 = ConvBlock(in_ch, base_ch)
        self.down1 = nn.Conv2d(base_ch, base_ch, 3, stride=2, padding=1)
        
        self.enc2 = ConvBlock(base_ch, base_ch * 2)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 3, stride=2, padding=1)
        
        # ── Bottleneck ────────────────────────────────────────────────
        self.bottleneck = ConvBlock(base_ch * 2, base_ch * 2)
        
        # ── Decoder ──────────────────────────────────────────────────
        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, 2, stride=2)
        self.dec1 = ConvBlock(base_ch * 2, base_ch)   # ×2 from skip connection
        
        self.up2 = nn.ConvTranspose2d(base_ch, base_ch, 2, stride=2)
        self.dec2 = ConvBlock(base_ch * 2, base_ch)   # ×2 from skip connection
        
        # ── Output head ──────────────────────────────────────────────
        self.out_conv = nn.Conv2d(base_ch, 1, 1)
        
        # Latent vector size after Global Average Pooling
        self.latent_dim = base_ch * 2

    def forward(self, x):
        e1 = self.enc1(x)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        d2 = self.down2(e2)

        b = self.bottleneck(d2)

        u1 = self.up1(b)
        u1 = torch.cat([u1, e2], dim=1)
        d1 = self.dec1(u1)

        u2 = self.up2(d1)
        u2 = torch.cat([u2, e1], dim=1)
        d2 = self.dec2(u2)

        return self.out_conv(d2)

    def get_latent(self, x):
        """
        Returns the latent VECTOR obtained by
        averaging the bottleneck feature map
        over spatial dimensions.
        """
        e1 = self.enc1(x)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        d2 = self.down2(e2)

        b = self.bottleneck(d2)          # (B, C, H/4, W/4)
        z = b.mean(dim=(2, 3))            # Global Average Pooling → (B, C)
        return z

    def get_latent_map(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns the bottleneck feature map WITHOUT global average pooling.
        Preserves spatial structure for use by CNN-based counters.

        Shape: (B, base_ch*2, H/4, W/4)
        For default settings (base_ch=32, input 256x256): (B, 64, 64, 64).
        """
        e1 = self.enc1(x)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        d2 = self.down2(e2)

        return self.bottleneck(d2)       # (B, base_ch*2, H/4, W/4)                       # (B, 1, 256, 256) logits

In [ ]:
unet = UNet(in_ch=1, base_ch=32)
print(unet)
print(f"\nBottleneck channels (latent_dim): {unet.latent_dim}")

#### 3.2 Training the U-Net

The U-Net is trained with a combination of two loss functions:

1. **Binary Cross-Entropy (BCE)** — treats each pixel independently, comparing the predicted probability to the ground-truth label (0 or 1)
2. **Dice Loss** — measures the overall overlap between the predicted and true masks; particularly helpful when classes are imbalanced

Together, these losses complement each other: BCE provides stable per-pixel gradients, while Dice Loss directly optimises for the region-level overlap metric we care about.

> **Note:** The `label` from each batch is ignored during U-Net training — segmentation only needs `(image, mask)` pairs. The label will be used later for classifier training.

In [ ]:
def train_unet(model, dataloader, epochs=50, lr=1e-3):
    """
    Train the U-Net segmentation model.
    
    Uses BCE + Dice loss. The dataloader yields (img, mask, label) tuples;
    the label is ignored here (only needed for classifier training).
    
    Args:
        model:      UNet instance
        dataloader: yields (images, masks, labels) batches
        epochs:     number of training epochs
        lr:         learning rate for Adam optimiser
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    
    bce = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for ep in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        for img, mask, _label in dataloader:  # _label unused for segmentation
            img  = img.to(device)
            mask = mask.to(device)
            
            pred = model(img)
            loss = bce(pred, mask) + dice_loss(pred, mask).mean()   # uses our utility function
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if ep % 10 == 0:
            avg = epoch_loss / len(dataloader)
            print(f"[U-Net] Epoch {ep:3d}/{epochs} | Loss: {avg:.4f}")

# Train the U-Net on the training set only
train_unet(unet, dl_train, epochs=50, lr=1e-3)

#### What to look for

- Loss should steadily decrease over the 50 epochs
- With 3500 training samples, the model should generalize well
- The segmentation should accurately identify intact vs damaged tile regions
- We're training on 70% of the data and will evaluate on the held-out 15% test set

In [ ]:
# Visualize U-Net predictions on TEST SET samples (unseen during training)
unet.eval()
unet_device = next(unet.parameters()).device

test_sample_indices = list(ds_test.indices[:8])  # First 8 from test set

fig, axes = plt.subplots(3, 8, figsize=(18, 9))
for col, idx in enumerate(test_sample_indices):
    img, mask, label = ds_full[idx]  # Now returns (img, mask, label)
    img_t = img.unsqueeze(0).to(unet_device)

    with torch.no_grad():
        pred = torch.sigmoid(unet(img_t)).squeeze().cpu()
    pred_bin = (pred > 0.5).float()
    
    status = "APPROVE" if label == 1 else "REJECT"
    coverage = ground_truth.iloc[idx]["coverage_pct"] * 100

    axes[0, col].imshow(img.squeeze(), cmap="gray")
    axes[0, col].set_title(f"mosaic_{idx+1:04d}\n{status}", fontsize=8)
    axes[0, col].axis("off")

    axes[1, col].imshow(mask.squeeze(), cmap="gray")
    axes[1, col].set_title(f"GT ({coverage:.1f}%)", fontsize=8)
    axes[1, col].axis("off")

    axes[2, col].imshow(pred_bin, cmap="gray")
    axes[2, col].set_title("U-Net pred", fontsize=8)
    axes[2, col].axis("off")

for row, lbl in enumerate(["Original", "GT mask", "U-Net pred"]):
    axes[row, 0].set_ylabel(lbl, fontsize=11)

plt.suptitle("U-Net segmentation results on TEST SET (unseen data)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4 🔢 FNN Classifier — Pooled Latent Vector

The first classification approach collapses the bottleneck into a single vector via **Global Average Pooling** and feeds it to a small feedforward network.

```
Bottleneck map (B, 64, 64, 64)
        ↓  Global Average Pooling  [inside UNet.get_latent()]
   Latent vector (B, 64)
        ↓  Linear(64 → 64) + ReLU
        ↓  Linear(64 → 1) + Sigmoid
   Approval probability p ∈ [0, 1]
```

**Binary Classification with Sigmoid**  
The final layer uses a **Sigmoid** activation, which squashes the output to the range $[0, 1]$. This is the natural choice for binary classification: the output directly represents the probability that the batch should be APPROVED.

We train with **Binary Cross-Entropy (BCE)** loss: the ground truth is 1 (APPROVE) if intact coverage ≥ 40%, or 0 (REJECT) otherwise.

**Architecture summary:**

| Layer | Input shape | Output shape |
|---|---|---|
| `Linear(64 → 64)` + ReLU | (B, 64) | (B, 64) |
| `Linear(64 → 1)` + Sigmoid | (B, 64) | (B,) → probability in [0, 1] |

#### 4.1 Architecture & Training

In [ ]:
class ApprovalClassifierFNN(nn.Module):
    """
    Feedforward classifier on the pooled latent vector.
    
    Takes the 64-dim vector from UNet.get_latent() and predicts
    the probability that a tile batch should be APPROVED.
    
    Architecture:
        Linear(64 → 64) + ReLU    — hidden layer
        Linear(64 → 1)  + Sigmoid — output probability ∈ [0, 1]
    
    The Sigmoid activation is the correct choice for binary classification:
    it maps any real number to (0, 1), giving us a proper probability.
    """
    
    def __init__(self, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, z):
        return self.net(z).squeeze(-1)   # (B,) probabilities


def train_classifier_fnn(unet, classifier, dataloader, epochs=50, lr=1e-3):
    """
    Train the FNN approval classifier.
    
    The U-Net is frozen (eval mode): we only train the classifier head.
    Labels come directly from the dataloader — no external list needed.
    
    Args:
        unet:       Trained U-Net (frozen, used for feature extraction)
        classifier: ApprovalClassifierFNN instance
        dataloader: yields (images, masks, labels) batches
        epochs:     number of training epochs
        lr:         learning rate for Adam optimiser
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    unet.eval()
    classifier.to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=lr)
    
    for ep in range(epochs):
        total_loss = 0.0
        for imgs, _masks, labels in dataloader:
            imgs   = imgs.to(device)
            labels = labels.float().to(device)
            
            with torch.no_grad():
                z = unet.get_latent(imgs)         # (B, 64)
            
            pred = classifier(z)                   # (B,) — already in [0, 1]
            loss = criterion(pred, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if ep % 10 == 0:
            avg = total_loss / len(dataloader)
            print(f"[FNN] Epoch {ep:3d}/{epochs} | Loss: {avg:.4f}")


# Instantiate and train
classifier_fnn = ApprovalClassifierFNN(latent_dim=unet.latent_dim)
train_classifier_fnn(unet, classifier_fnn, dl_train, epochs=50)

---
## 5 🧠 CNN Classifier — Spatial Latent Map

The second approach skips the early pooling step. Instead of a 64-dimensional vector, the classifier receives the **full spatial feature map** `(B, 64, 64, 64)` and processes it with convolutional layers before any aggregation.

**Why might spatial structure help for classification?**

Global Average Pooling discards *where* activations are located. For surface area assessment, spatial structure is directly informative:
- The spatial distribution of intact regions tells us about coverage patterns
- Edge effects, clustering, and spatial connectivity matter for quality assessment
- Convolutional filters can detect patterns like "scattered small intact patches" vs "large contiguous intact areas"

**Architecture:**

| Layer | Input shape | Output shape | Purpose |
|---|---|---|---|
| `Conv2d(64→32, 3×3, pad=1)` + BN + ReLU | (B,64,64,64) | (B,32,64,64) | Detect local patterns in bottleneck |
| `MaxPool2d(2)` | (B,32,64,64) | (B,32,32,32) | Halve spatial size, widen receptive field |
| `Conv2d(32→16, 3×3, pad=1)` + BN + ReLU | (B,32,32,32) | (B,16,32,32) | Higher-level spatial features |
| `AdaptiveAvgPool2d(1)` (GAP) | (B,16,32,32) | (B,16,1,1) | Aggregate spatially |
| `Flatten` → `Linear(16→1)` → `Sigmoid` | (B,16) | (B,) | Probability in [0, 1] |

The GAP here sits **after** two convolutional layers — the key difference from the FNN, where pooling happens *before* any learned processing. This allows the CNN to first learn what spatial patterns matter, then summarise.

#### 5.1 Architecture & Training

In [ ]:
class ApprovalClassifierCNN(nn.Module):
    """
    Convolutional classifier on the spatial latent map.
    
    Takes the (B, 64, 64, 64) feature map from UNet.get_latent_map()
    and predicts the probability that a tile batch should be APPROVED.
    
    Architecture:
        Conv2d(64→32) + BN + ReLU     — detect local patterns
        MaxPool2d(2)                   — halve spatial resolution
        Conv2d(32→16) + BN + ReLU     — higher-level features
        AdaptiveAvgPool2d(1)           — global aggregation
        Flatten → Linear(16→1) + Sigmoid  — probability ∈ [0, 1]
    
    Unlike the FNN, this classifier preserves spatial structure through
    two convolutional stages before pooling. This allows it to learn
    spatially-aware features (e.g., coverage distribution, contiguity).
    """
    
    def __init__(self, in_channels=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, z_map):
        x = self.features(z_map)
        return self.classifier(x).squeeze(-1)   # (B,) probabilities


def train_classifier_cnn(unet, classifier, dataloader, epochs=50, lr=1e-3):
    """
    Train the CNN approval classifier.
    
    The U-Net is frozen (eval mode): we only train the classifier head.
    Uses the spatial latent map rather than the pooled vector.
    
    Args:
        unet:       Trained U-Net (frozen, used for feature extraction)
        classifier: ApprovalClassifierCNN instance
        dataloader: yields (images, masks, labels) batches
        epochs:     number of training epochs
        lr:         learning rate for Adam optimiser
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    unet.eval()
    classifier.to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=lr)
    
    for ep in range(epochs):
        total_loss = 0.0
        for imgs, _masks, labels in dataloader:
            imgs   = imgs.to(device)
            labels = labels.float().to(device)
            
            with torch.no_grad():
                z_map = unet.get_latent_map(imgs)  # (B, 64, 64, 64)
            
            pred = classifier(z_map)                # (B,) — already in [0, 1]
            loss = criterion(pred, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if ep % 10 == 0:
            avg = total_loss / len(dataloader)
            print(f"[CNN] Epoch {ep:3d}/{epochs} | Loss: {avg:.4f}")


# Instantiate and train
classifier_cnn = ApprovalClassifierCNN(in_channels=unet.latent_dim)
train_classifier_cnn(unet, classifier_cnn, dl_train, epochs=50)

---
## 6 ⚖️ Comparison: FNN vs CNN Classifier

We now evaluate both classifiers on the **TEST SET** (held-out data) and compare their predictions against the ground truth. We also include a **direct threshold** baseline that calculates coverage from the U-Net segmentation mask (if >40% white pixels → APPROVE).

In [ ]:
unet.eval()
classifier_fnn.eval()
classifier_cnn.eval()

# Evaluate on ALL test set samples (unseen during training)
rows = []
for idx in ds_test.indices:
    img, mask, label = ds_full[idx]  # Now includes label
    img_t = img.unsqueeze(0).to(unet_device)

    with torch.no_grad():
        # Segmentation mask → direct threshold
        logits = unet(img_t)
        seg_mask = torch.sigmoid(logits)
        coverage = seg_mask.mean().item()
        threshold_pred = "APPROVE" if coverage > 0.4 else "REJECT"

        # FNN classifier: pooled latent vector
        z = unet.get_latent(img_t)
        fnn_prob = classifier_fnn(z).item()
        fnn_pred = "APPROVE" if fnn_prob > 0.5 else "REJECT"

        # CNN classifier: spatial latent map
        z_map = unet.get_latent_map(img_t)
        cnn_prob = classifier_cnn(z_map).item()
        cnn_pred = "APPROVE" if cnn_prob > 0.5 else "REJECT"

    actual = "APPROVE" if label == 1 else "REJECT"
    actual_coverage = ground_truth.iloc[idx]["coverage_pct"]
    
    rows.append({
        "Image":       f"mosaic_{idx+1:04d}",
        "Coverage":    f"{actual_coverage*100:.1f}%",
        "Actual":      actual,
        "Threshold":   threshold_pred,
        "FNN pred":    fnn_pred,
        "CNN pred":    cnn_pred,
    })

# Show first 20 results
df = pd.DataFrame(rows)
print(f"Evaluating on {len(df)} test samples (unseen during training)\n")
df.head(20).set_index("Image")

In [ ]:
# Calculate accuracy for each method on the full test set
threshold_correct = sum(df["Actual"] == df["Threshold"])
fnn_correct = sum(df["Actual"] == df["FNN pred"])
cnn_correct = sum(df["Actual"] == df["CNN pred"])

methods = ["Threshold", "FNN", "CNN"]
accuracies = [
    threshold_correct / len(df) * 100,
    fnn_correct / len(df) * 100,
    cnn_correct / len(df) * 100
]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(methods, accuracies, color=["goldenrod", "tomato", "seagreen"])
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"Batch Approval Classification on Test Set (n={len(df)} held-out samples)")
ax.set_ylim([0, 105])

# Add accuracy labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print(f"TEST SET ACCURACY (n={len(df)} unseen samples)")
print(f"{'='*60}")
print(f"  Threshold (40% coverage): {accuracies[0]:.1f}% ({threshold_correct}/{len(df)} correct)")
print(f"  FNN classifier:           {accuracies[1]:.1f}% ({fnn_correct}/{len(df)} correct)")
print(f"  CNN classifier:           {accuracies[2]:.1f}% ({cnn_correct}/{len(df)} correct)")
print(f"{'='*60}")

#### Interpreting the comparison

| Method | What it sees | Inductive bias |
|---|---|---|
| **Threshold (40%)** | Predicted segmentation mask | Purely rule-based: count white pixels, apply 40% cutoff |
| **FNN classifier** | 64-dim pooled vector | Must encode all surface area information in a fixed-size summary |
| **CNN classifier** | 64×64×64 spatial map | Can learn spatial patterns before aggregating |

**Why this comparison matters:**

The **Threshold** baseline is deterministic: it directly implements the business rule. If the U-Net segmentation is perfect, the threshold achieves 100% accuracy.

The **FNN classifier** learns a non-linear mapping from the pooled bottleneck to approval/rejection. It can compensate for U-Net segmentation errors, but it has no access to spatial structure.

The **CNN classifier** is the most flexible: it sees the spatial distribution of intact regions and can learn patterns that go beyond simple coverage (e.g., "large contiguous areas are more valuable than scattered patches").

With a large dataset (5000 samples), we expect:
- The threshold to work well if U-Net segmentation is accurate
- The CNN to potentially outperform both by learning subtle spatial patterns
- The FNN to fall between the two, as it has limited spatial awareness

---
## 7 🚀 Full Pipeline Inference on a Single Image

Let us walk through the entire pipeline on a sample batch from the **test set** near the 40% threshold — the most challenging case where the decision matters most.

In [ ]:
# Find a batch near the 40% threshold from the TEST SET for demonstration
threshold_test_samples = []
for idx in ds_test.indices:
    cov = ground_truth.iloc[idx]["coverage_pct"]
    if 0.35 <= cov <= 0.45:
        threshold_test_samples.append((idx, cov))

# Pick one near the middle
IDX = threshold_test_samples[len(threshold_test_samples)//2][0] if threshold_test_samples else ds_test.indices[0]

img, gt_mask, label = ds_full[IDX]  # Now includes label
img_t = img.unsqueeze(0).to(unet_device)

unet.eval()
classifier_fnn.eval()
classifier_cnn.eval()

with torch.no_grad():
    # Step 1: segmentation
    logits = unet(img_t)
    seg_pred = torch.sigmoid(logits)
    seg_bin = (seg_pred > 0.5).float()
    coverage = seg_pred.mean().item()
    threshold_decision = "APPROVE" if coverage > 0.4 else "REJECT"

    # Step 2: FNN path — pooled latent vector
    z = unet.get_latent(img_t)                     # (1, 64)
    fnn_prob = classifier_fnn(z).item()
    fnn_decision = "APPROVE" if fnn_prob > 0.5 else "REJECT"

    # Step 3: CNN path — spatial latent map
    z_map = unet.get_latent_map(img_t)             # (1, 64, 64, 64)
    cnn_prob = classifier_cnn(z_map).item()
    cnn_decision = "APPROVE" if cnn_prob > 0.5 else "REJECT"

actual_decision = "APPROVE" if label == 1 else "REJECT"
actual_coverage = ground_truth.iloc[IDX]["coverage_pct"]

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img.squeeze(), cmap="gray")
axes[0].set_title(f"Original: mosaic_{IDX+1:04d}", fontsize=11)
axes[1].imshow(gt_mask.squeeze(), cmap="gray")
axes[1].set_title(f"Ground truth mask\n{actual_coverage*100:.1f}% intact", fontsize=11)
axes[2].imshow(seg_bin.squeeze().cpu(), cmap="gray")
axes[2].set_title(f"U-Net prediction\n{coverage*100:.1f}% intact", fontsize=11)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"mosaic_{IDX+1:04d} — Full pipeline (TEST SET sample near 40%)", fontsize=13)
plt.tight_layout()
plt.show()

print("=" * 60)
print("INFERENCE SUMMARY (TEST SET SAMPLE)")
print("=" * 60)
print(f"Batch:                mosaic_{IDX+1:04d}")
print(f"Actual coverage:      {actual_coverage*100:.1f}%")
print(f"Actual decision:      {actual_decision}")
print("-" * 60)
print(f"U-Net coverage:       {coverage*100:.1f}%")
print(f"Threshold decision:   {threshold_decision}  (>40% → APPROVE)")
print(f"FNN decision:         {fnn_decision}  (prob={fnn_prob:.3f})")
print(f"CNN decision:         {cnn_decision}  (prob={cnn_prob:.3f})")
print("=" * 60)

#### Takeaways

**Three classification strategies, three philosophies:**

- **Threshold (40% rule)** is a **deterministic** approach — purely rule-based. It counts white pixels in the U-Net mask and applies the 40% cutoff. Fast and interpretable, but completely dependent on segmentation quality. If the U-Net mis-segments, the threshold fails.

- The **FNN classifier** collapses the entire bottleneck into a 64-number summary via Global Average Pooling, then runs a simple two-layer MLP. It learns *how much* signal is in the bottleneck on average, but loses information about *where* intact regions appear. It can learn to compensate for systematic U-Net errors.

- The **CNN classifier** processes the bottleneck as a spatial feature map, using two convolutional layers before aggregating. It can detect spatial patterns (e.g., "large contiguous intact area" vs "scattered small patches") before collapsing to a decision. This is the architecturally richer approach.

> **Key insight:** The choice between FNN and CNN for the classification head is a trade-off between *model simplicity* and *information preservation*. When the input is inherently spatial and the task depends on spatial structure — as surface area quality assessment does — preserving that structure through convolutional processing before aggregation is the better design.

**Business impact:**  
For the recycling warehouse, accurate batch approval is critical:
- **False positives** (approving bad batches) → unhappy customers, returns
- **False negatives** (rejecting good batches) → wasted manual inspection time, lost sales

The CNN classifier's ability to understand spatial coverage patterns makes it the strongest candidate for production deployment.